## 1. Imports and Paths

In [1]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"

LEADS_FILE = RAW_DATA_DIR / "leads.csv"
EVENTS_FILE = RAW_DATA_DIR / "lead_stage_events.csv"

print(f"Project root: {ROOT_DIR}")
print(f"Leads file: {LEADS_FILE}")
print(f"Events file: {EVENTS_FILE}")


Project root: c:\Projects\b2b-growth-funnel-analytics
Leads file: c:\Projects\b2b-growth-funnel-analytics\data\raw\leads.csv
Events file: c:\Projects\b2b-growth-funnel-analytics\data\raw\lead_stage_events.csv


## 2. Load Raw Data

In [2]:
leads = pd.read_csv(LEADS_FILE)
events = pd.read_csv(EVENTS_FILE)

print(f"Leads shape: {leads.shape}")
print(f"Events shape: {events.shape}")


Leads shape: (500, 8)
Events shape: (1445, 4)


## 3. Preview the Datasets

In [3]:
leads.head()

,lead_id,created_date,channel,campaign,region,company_size,industry,acquisition_cost
0,L00001,2025-03-25,Webinar,Finance Automation Webinar,US,Enterprise,Healthcare,38.15
1,L00002,2025-06-11,Email,Product Update,US,Enterprise,Healthcare,26.38
2,L00003,2025-04-10,Organic Search,SEO Evergreen,US,SMB,Retail,12.34
3,L00004,2025-06-02,Organic Search,Blog CTA,US,SMB,Logistics,11.88
4,L00005,2025-04-29,Referral,Customer Referral,UK,Mid-Market,Healthcare,9.08


In [4]:
events.head()

,lead_id,stage,stage_date,revenue
0,L00001,Lead,2025-03-25,0.0
1,L00001,Lost,2025-04-04,0.0
2,L00002,Lead,2025-06-11,0.0
3,L00002,MQL,2025-06-24,0.0
4,L00002,Lost,2025-07-13,0.0


## 4. Structure and Data Types

In [5]:
print("LEADS DATASET")
print("-" * 50)
print(f"Rows: {len(leads):,}")
print(f"Columns: {leads.shape[1]}")
print("\nColumns:")
print(leads.columns.tolist())
print("\nData types:")
print(leads.dtypes)


LEADS DATASET
--------------------------------------------------
Rows: 500
Columns: 8

Columns:
['lead_id', 'created_date', 'channel', 'campaign', 'region', 'company_size', 'industry', 'acquisition_cost']

Data types:
lead_id                 str
created_date            str
channel                 str
campaign                str
region                  str
company_size            str
industry                str
acquisition_cost    float64
dtype: object


In [6]:
print("FUNNEL EVENTS DATASET")
print("-" * 50)
print(f"Rows: {len(events):,}")
print(f"Columns: {events.shape[1]}")
print("\nColumns:")
print(events.columns.tolist())
print("\nData types:")
print(events.dtypes)


FUNNEL EVENTS DATASET
--------------------------------------------------
Rows: 1,445
Columns: 4

Columns:
['lead_id', 'stage', 'stage_date', 'revenue']

Data types:
lead_id           str
stage             str
stage_date        str
revenue       float64
dtype: object


## 5. Missing Values

In [7]:
lead_missing = pd.DataFrame({
    "missing_count": leads.isna().sum(),
    "missing_pct": (leads.isna().mean() * 100).round(2),
})
lead_missing


,missing_count,missing_pct
lead_id,0,0.0
created_date,0,0.0
channel,0,0.0
campaign,0,0.0
region,0,0.0
company_size,0,0.0
industry,0,0.0
acquisition_cost,0,0.0


In [8]:
event_missing = pd.DataFrame({
    "missing_count": events.isna().sum(),
    "missing_pct": (events.isna().mean() * 100).round(2),
})
event_missing


,missing_count,missing_pct
lead_id,0,0.0
stage,0,0.0
stage_date,0,0.0
revenue,0,0.0


## 6. Duplicate Checks

In [9]:
print(f"Duplicate rows in leads: {leads.duplicated().sum():,}")
print(f"Duplicate rows in events: {events.duplicated().sum():,}")
print(f"Duplicate lead IDs: {leads['lead_id'].duplicated().sum():,}")


Duplicate rows in leads: 0
Duplicate rows in events: 0
Duplicate lead IDs: 0


## 7. Business Categorical Fields

In [10]:
categorical_columns = [
    "channel",
    "campaign",
    "region",
    "company_size",
    "industry",
]

for column in categorical_columns:
    print(f"\n{column.upper()}")
    print("-" * 40)
    print(leads[column].value_counts(dropna=False).sort_index())



CHANNEL
----------------------------------------
channel
Email             111
Google Ads         90
LinkedIn           76
Organic Search     88
Referral           83
Webinar            52
Name: count, dtype: int64

CAMPAIGN
----------------------------------------
campaign
ABM Decision Makers              28
Blog CTA                         30
Brand Campaign                   34
Comparison Pages                 23
Competitor Terms                 29
Customer Referral                48
Finance Automation Webinar       27
Finance Leaders                  16
Nurture Flow                     35
Ops Leaders                      32
Partner Referral                 35
Product Update                   38
Reactivation                     38
SEO Evergreen                    35
Search Campaign                  27
Workflow Optimization Session    25
Name: count, dtype: int64

REGION
----------------------------------------
region
Canada    119
UK         75
US        306
Name: count, dtype: int6

In [11]:
print("FUNNEL STAGES")
print("-" * 40)
print(events["stage"].value_counts(dropna=False))


FUNNEL STAGES
----------------------------------------
stage
Lead        500
Lost        452
MQL         298
SQL         147
Customer     48
Name: count, dtype: int64


## 8. Numerical Fields

In [12]:
leads["acquisition_cost"].describe()

count    500.000000
mean      39.814480
std       34.206983
min        3.180000
25%       11.487500
50%       23.030000
75%       66.332500
max      129.420000
Name: acquisition_cost, dtype: float64

In [13]:
events["revenue"].describe()

count     1445.000000
mean       405.263419
std       2435.113091
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      25873.550000
Name: revenue, dtype: float64

In [14]:
negative_acquisition_cost = (leads["acquisition_cost"] < 0).sum()
negative_revenue = (events["revenue"] < 0).sum()

print(f"Negative acquisition-cost records: {negative_acquisition_cost:,}")
print(f"Negative revenue records: {negative_revenue:,}")


Negative acquisition-cost records: 0
Negative revenue records: 0


## 9. Date Validation and Coverage

In [15]:
lead_dates = pd.to_datetime(leads["created_date"], errors="coerce")
event_dates = pd.to_datetime(events["stage_date"], errors="coerce")

print(f"Lead date range: {lead_dates.min().date()} → {lead_dates.max().date()}")
print(f"Event date range: {event_dates.min().date()} → {event_dates.max().date()}")
print()
print(f"Invalid lead dates: {lead_dates.isna().sum():,}")
print(f"Invalid event dates: {event_dates.isna().sum():,}")


Lead date range: 2025-01-01 → 2025-06-30
Event date range: 2025-01-01 → 2025-08-20

Invalid lead dates: 0
Invalid event dates: 0


## 10. Key and Relationship Checks

In [16]:
orphan_events = events.loc[
    ~events["lead_id"].isin(leads["lead_id"])
]

leads_without_events = leads.loc[
    ~leads["lead_id"].isin(events["lead_id"])
]

print(f"Unique lead IDs in leads: {leads['lead_id'].nunique():,}")
print(f"Unique lead IDs in events: {events['lead_id'].nunique():,}")
print(f"Orphan event records: {len(orphan_events):,}")
print(f"Leads without any funnel events: {len(leads_without_events):,}")


Unique lead IDs in leads: 500
Unique lead IDs in events: 500
Orphan event records: 0
Leads without any funnel events: 0


## 11. Funnel Summary

In [17]:
total_leads = leads["lead_id"].nunique()

stage_counts = (
    events.groupby("stage")["lead_id"]
    .nunique()
    .reindex(["Lead", "MQL", "SQL", "Customer", "Lost"])
)

customers = events.loc[
    events["stage"] == "Customer", "lead_id"
].nunique()

lost_leads = events.loc[
    events["stage"] == "Lost", "lead_id"
].nunique()

overall_conversion_rate = customers / total_leads * 100 if total_leads else 0

print(f"Total leads: {total_leads:,}")
print("\nUnique leads by stage:")
print(stage_counts)
print()
print(f"Customers: {customers:,}")
print(f"Lost leads: {lost_leads:,}")
print(f"Lead-to-customer conversion rate: {overall_conversion_rate:.2f}%")


Total leads: 500

Unique leads by stage:
stage
Lead        500
MQL         298
SQL         147
Customer     48
Lost        452
Name: lead_id, dtype: int64

Customers: 48
Lost leads: 452
Lead-to-customer conversion rate: 9.60%


## 12. Stage-to-Stage Conversion Rates

In [18]:
lead_count = stage_counts.get("Lead", 0)
mql_count = stage_counts.get("MQL", 0)
sql_count = stage_counts.get("SQL", 0)
customer_count = stage_counts.get("Customer", 0)

lead_to_mql = mql_count / lead_count * 100 if lead_count else 0
mql_to_sql = sql_count / mql_count * 100 if mql_count else 0
sql_to_customer = customer_count / sql_count * 100 if sql_count else 0

funnel_rates = pd.DataFrame({
    "transition": [
        "Lead → MQL",
        "MQL → SQL",
        "SQL → Customer",
        "Lead → Customer",
    ],
    "conversion_rate_pct": [
        lead_to_mql,
        mql_to_sql,
        sql_to_customer,
        overall_conversion_rate,
    ],
})

funnel_rates["conversion_rate_pct"] = (
    funnel_rates["conversion_rate_pct"].round(2)
)

funnel_rates


,transition,conversion_rate_pct
0,Lead → MQL,59.60
1,MQL → SQL,49.33
2,SQL → Customer,32.65
3,Lead → Customer,9.60


## 13. Revenue Consistency

In [19]:
customer_events = events.loc[events["stage"] == "Customer"]
non_customer_events = events.loc[events["stage"] != "Customer"]

customers_without_revenue = (
    customer_events["revenue"].fillna(0) <= 0
).sum()

non_customers_with_revenue = (
    non_customer_events["revenue"].fillna(0) > 0
).sum()

total_customer_revenue = customer_events["revenue"].fillna(0).sum()

print(f"Customer events without positive revenue: {customers_without_revenue:,}")
print(f"Non-customer events with positive revenue: {non_customers_with_revenue:,}")
print(f"Total customer revenue: {total_customer_revenue:,.2f}")


Customer events without positive revenue: 0
Non-customer events with positive revenue: 0
Total customer revenue: 585,605.64


## 14. Events Before Lead Creation

In [20]:
date_check = events.copy()
date_check["stage_date"] = pd.to_datetime(
    date_check["stage_date"], errors="coerce"
)

lead_date_lookup = leads[["lead_id", "created_date"]].copy()
lead_date_lookup["created_date"] = pd.to_datetime(
    lead_date_lookup["created_date"], errors="coerce"
)

date_check = date_check.merge(
    lead_date_lookup,
    on="lead_id",
    how="left",
)

events_before_creation = date_check.loc[
    date_check["stage_date"] < date_check["created_date"]
]

print(f"Events before lead creation: {len(events_before_creation):,}")


Events before lead creation: 0


## 15. Funnel Stage Sequence Validation

Expected successful progression:

**Lead → MQL → SQL → Customer**

`Lost` is treated as a terminal outcome and excluded from ordinal progression comparisons.


In [21]:
stage_order = {
    "Lead": 1,
    "MQL": 2,
    "SQL": 3,
    "Customer": 4,
}

sequence_check = events.copy()
sequence_check["stage_date"] = pd.to_datetime(
    sequence_check["stage_date"], errors="coerce"
)

invalid_sequence_leads = []

for lead_id, lead_events in sequence_check.groupby("lead_id"):
    ordered_events = lead_events.sort_values("stage_date")
    previous_rank = 0

    for stage in ordered_events["stage"]:
        if stage == "Lost":
            continue

        current_rank = stage_order.get(stage)

        if current_rank is None:
            continue

        if current_rank < previous_rank:
            invalid_sequence_leads.append(lead_id)
            break

        previous_rank = current_rank

print(f"Leads with out-of-order funnel stages: {len(invalid_sequence_leads):,}")


Leads with out-of-order funnel stages: 0


## 16. Terminal-State Checks

In [22]:
terminal_stages = events.loc[
    events["stage"].isin(["Customer", "Lost"]),
    ["lead_id", "stage"],
]

terminal_stage_counts = (
    terminal_stages.groupby("lead_id")["stage"].nunique()
)

multiple_terminal_outcomes = terminal_stage_counts[
    terminal_stage_counts > 1
]

print(
    "Leads with both Customer and Lost outcomes: "
    f"{len(multiple_terminal_outcomes):,}"
)


Leads with both Customer and Lost outcomes: 0


## 17. High-Level Acquisition Economics

In [23]:
total_acquisition_spend = leads["acquisition_cost"].sum()
total_revenue = customer_events["revenue"].sum()

cac = (
    total_acquisition_spend / customers
    if customers
    else 0
)

roas = (
    total_revenue / total_acquisition_spend
    if total_acquisition_spend
    else 0
)

revenue_per_lead = (
    total_revenue / total_leads
    if total_leads
    else 0
)

print(f"Acquisition spend: {total_acquisition_spend:,.2f}")
print(f"Customer revenue: {total_revenue:,.2f}")
print(f"CAC: {cac:,.2f}")
print(f"ROAS: {roas:.2f}x")
print(f"Revenue per lead: {revenue_per_lead:,.2f}")


Acquisition spend: 19,907.24
Customer revenue: 585,605.64
CAC: 414.73
ROAS: 29.42x
Revenue per lead: 1,171.21


## 18. Audit Summary

In [24]:
audit_summary = pd.DataFrame({
    "check": [
        "Lead rows",
        "Event rows",
        "Duplicate lead IDs",
        "Orphan event records",
        "Leads without events",
        "Invalid lead dates",
        "Invalid event dates",
        "Negative acquisition costs",
        "Negative revenue records",
        "Events before lead creation",
        "Invalid stage sequences",
        "Both Customer and Lost outcomes",
        "Customers without positive revenue",
        "Non-customers with positive revenue",
    ],
    "value": [
        len(leads),
        len(events),
        leads["lead_id"].duplicated().sum(),
        len(orphan_events),
        len(leads_without_events),
        lead_dates.isna().sum(),
        event_dates.isna().sum(),
        negative_acquisition_cost,
        negative_revenue,
        len(events_before_creation),
        len(invalid_sequence_leads),
        len(multiple_terminal_outcomes),
        customers_without_revenue,
        non_customers_with_revenue,
    ],
})

audit_summary


,check,value
0,Lead rows,500
1,Event rows,1445
2,Duplicate lead IDs,0
3,Orphan event records,0
4,Leads without events,0
5,Invalid lead dates,0
6,Invalid event dates,0
7,Negative acquisition costs,0
8,Negative revenue records,0
9,Events before lead creation,0
